# جریان کاری انسانی-در-حلقه با چارچوب Agent مایکروسافت

## 🎯 اهداف یادگیری

در این دفترچه یادداشت، خواهید آموخت چگونه جریان‌های کاری **انسانی-در-حلقه** را با استفاده از `RequestInfoExecutor` چارچوب Agent مایکروسافت پیاده‌سازی کنید. این الگوی قدرتمند به شما اجازه می‌دهد جریان‌های کاری هوش مصنوعی را برای جمع‌آوری ورودی‌های انسانی متوقف کنید، و عوامل شما را تعاملی کرده و به انسان‌ها کنترل تصمیمات مهم را بدهد.

## 🔄 انسانی-در-حلقه چیست؟

**انسانی-در-حلقه (HITL)** یک الگوی طراحی است که در آن عوامل هوش مصنوعی اجرای خود را متوقف می‌کنند تا قبل از ادامه، ورودی انسان را درخواست کنند. این موضوع برای موارد زیر ضروری است:

- ✅ **تصمیمات حساس** - دریافت تأیید انسان قبل از انجام اقدامات مهم
- ✅ **موقعیت‌های مبهم** - اجازه دادن به انسان‌ها برای روشن‌سازی زمانی که هوش مصنوعی مطمئن نیست
- ✅ **ترجیحات کاربر** - درخواست از کاربران برای انتخاب بین گزینه‌های متعدد
- ✅ **انطباق و ایمنی** - اطمینان از نظارت انسانی بر عملیات‌های قانونمند
- ✅ **تجارب تعاملی** - ساخت عوامل مکالمه‌ای که به ورودی کاربران پاسخ می‌دهند

## 🏗️ چگونه در چارچوب Agent مایکروسافت کار می‌کند

چارچوب سه جزء کلیدی برای HITL فراهم می‌کند:

1. **`RequestInfoExecutor`** - یک اجراکننده خاص که جریان کاری را متوقف می‌کند و رویداد `RequestInfoEvent` را ارسال می‌کند
2. **`RequestInfoMessage`** - کلاس پایه برای بار درخواست تایپ شده ارسالی به انسان‌ها
3. **`RequestResponse`** - پاسخ‌های انسان را با درخواست‌های اصلی با استفاده از `request_id` هم‌بسته می‌کند

**الگوی جریان کاری:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 مثال ما: رزرو هتل با تأیید کاربر

ما بر جریان کاری شرطی پایه خواهیم گذاشت و تأیید انسان را **قبل از** پیشنهاد مقصدهای جایگزین اضافه خواهیم کرد:

1. کاربر مقصدی درخواست می‌کند (مثلاً "پاریس")
2. `availability_agent` بررسی می‌کند که آیا اتاق‌ها موجود هستند
3. **اگر هیچ اتاقی نیست** → `confirmation_agent` می‌پرسد "آیا می‌خواهید جایگزین‌ها را ببینید؟"
4. جریان کاری با استفاده از `RequestInfoExecutor` **متوقف** می‌شود
5. **انسان پاسخ می‌دهد** "بله" یا "خیر" از طریق ورودی کنسول
6. `decision_manager` بر اساس پاسخ مسیر دهی می‌کند:
   - **بله** → نمایش مقصدهای جایگزین
   - **خیر** → لغو درخواست رزرو
7. نمایش نتیجه نهایی

این نشان می‌دهد چگونه به کاربران کنترل روی پیشنهادهای عامل داده شود!

---

بیایید شروع کنیم! 🚀


## گام ۱: وارد کردن کتابخانه‌های مورد نیاز

ما مؤلفه‌های استاندارد چارچوب عامل را به‌علاوه **کلاس‌های خاص انسان در حلقه** وارد می‌کنیم:
- `RequestInfoExecutor` - اجراکننده‌ای که جریان کار را برای ورودی انسان مکث می‌دهد
- `RequestInfoEvent` - رویدادی که هنگام درخواست ورودی انسان صادر می‌شود
- `RequestInfoMessage` - کلاس پایه برای بار درخواست تایپ‌شده
- `RequestResponse` - پاسخ‌های انسانی را با درخواست‌ها مرتبط می‌کند
- `WorkflowOutputEvent` - رویدادی برای شناسایی خروجی‌های جریان کار


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## مرحله ۲: تعریف مدل‌های Pydantic برای خروجی‌های ساخت‌یافته

این مدل‌ها **طرح‌واره**‌ای را تعریف می‌کنند که عوامل بازمی‌گردانند. ما همه مدل‌های جریان کاری شرطی را نگه می‌داریم و اضافه می‌کنیم:

**جدید برای انسان در حلقه:**
- `HumanFeedbackRequest` - زیرکلاسی از `RequestInfoMessage` که بار درخواستی ارسال شده به انسان‌ها را تعریف می‌کند
  - شامل `prompt` (سؤال برای پرسیدن) و `destination` (متنی درباره شهر غیرقابل دسترس)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## مرحله ۳: ایجاد ابزار رزرو هتل

همان ابزاری که در جریان کاری شرطی بود - بررسی می‌کند که آیا اتاق‌ها در مقصد در دسترس هستند یا خیر.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## مرحله ۴: تعریف توابع شرطی برای مسیر‌یابی

ما به **چهار تابع شرطی** برای جریان کار انسان-در-حلقه نیاز داریم:

**از جریان کار شرطی:**
۱. `has_availability_condition` - مسیر‌دهی زمانی که هتل‌ها موجود هستند
۲. `no_availability_condition` - مسیر‌دهی زمانی که هتل‌ها موجود نیستند

**جدید برای انسان-در-حلقه:**
۳. `user_wants_alternatives_condition` - مسیر‌دهی زمانی که کاربر به جایگزین‌ها "بله" می‌گوید
۴. `user_declines_alternatives_condition` - مسیر‌دهی زمانی که کاربر به جایگزین‌ها "خیر" می‌گوید


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## مرحله ۵: ایجاد اجرایی‌کننده مدیر تصمیم

این **هسته الگوی انسان در حلقه است**! `DecisionManager` یک `Executor` سفارشی است که:

۱. **بازخورد انسانی را از طریق اشیاء `RequestResponse` دریافت می‌کند**
۲. **تصمیم کاربر را پردازش می‌کند** (بله/خیر)
۳. **جریان کار را با ارسال پیام به عوامل مناسب هدایت می‌کند**

ویژگی‌های کلیدی:
- استفاده از دکوراتور `@handler` برای نمایش روش‌ها به عنوان مراحل جریان کار
- دریافت `RequestResponse[HumanFeedbackRequest, str]` که هم درخواست اصلی و هم پاسخ کاربر را در بر دارد
- تولید پیام‌های ساده «بله» یا «خیر» که توابع شرط ما را فعال می‌کنند


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## مرحله ۶: ایجاد اجرایی‌گر نمایش سفارشی

همان اجرایی‌گر نمایشی از جریان کاری شرطی - نتایج نهایی را به عنوان خروجی جریان کاری ارائه می‌دهد.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## مرحله ۷: بارگذاری متغیرهای محیطی

پیکربندی کلاینت LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## مرحله ۸: ایجاد عوامل هوش مصنوعی و مجریان

ما **شش مؤلفه‌ی جریان کاری** ایجاد می‌کنیم:

**عوامل (درون AgentExecutor پیچیده شده‌اند):**
۱. **availability_agent** - بررسی موجودی هتل با استفاده از ابزار
۲. **confirmation_agent** - 🆕 آماده‌سازی درخواست تأیید انسانی
۳. **alternative_agent** - پیشنهاد شهرهای جایگزین (وقتی کاربر بله می‌گوید)
۴. **booking_agent** - تشویق به رزرو (وقتی اتاق‌ها موجود هستند)
۵. **cancellation_agent** - 🆕 مدیریت پیام لغو (وقتی کاربر خیر می‌گوید)

**مجریان ویژه:**
۶. **request_info_executor** - 🆕 `RequestInfoExecutor` که جریان کاری را برای ورودی انسانی متوقف می‌کند
۷. **decision_manager** - 🆕 مجری سفارشی که بر اساس پاسخ انسانی مسیر را تعیین می‌کند (قبلاً در بالا تعریف شده است)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## مرحله ۹: ساخت گردش کار با دخالت انسان در حلقه

اکنون نمودار گردش کار را با **مسیرهای شرطی** شامل مسیر انسان در حلقه می‌سازیم:

**ساختار گردش کار:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**یال‌های کلیدی:**
- `availability_agent → confirmation_agent` (وقتی اتاقی نیست)
- `confirmation_agent → prepare_human_request` (تغییر نوع)
- `prepare_human_request → request_info_executor` (توقف برای دخالت انسان)
- `request_info_executor → decision_manager` (همیشه - پاسخ درخواست فراهم می‌کند)
- `decision_manager → alternative_agent` (وقتی کاربر می‌گوید "بله")
- `decision_manager → cancellation_agent` (وقتی کاربر می‌گوید "خیر")
- `availability_agent → booking_agent` (وقتی اتاق موجود است)
- همه مسیرها به `display_result` ختم می‌شوند


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## گام ۱۰: اجرای مورد آزمایشی ۱ - شهر بدون موجودی (پاریس با تأیید انسان)

این آزمایش **چرخه کامل انسان در حلقه** را نشان می‌دهد:

۱. درخواست هتل در پاریس
۲. بررسی availability_agent → عدم وجود اتاق
۳. confirmation_agent پرسش مواجه با انسان را ایجاد می‌کند
۴. request_info_executor **جریان کار را متوقف می‌کند** و `RequestInfoEvent` را ارسال می‌نماید
۵. **برنامه رویداد را شناسایی کرده و از کاربر در کنسول سوال می‌کند**
۶. کاربر "yes" یا "no" را تایپ می‌کند
۷. برنامه پاسخ را از طریق `send_responses_streaming()` ارسال می‌کند
۸. decision_manager بر اساس پاسخ مسیر دهی می‌کند
۹. نتیجه نهایی نمایش داده می‌شود

**الگوی کلیدی:**
- برای تکرار اول از `workflow.run_stream()` استفاده کنید
- برای تکرارهای بعدی از `workflow.send_responses_streaming(pending_responses)` استفاده کنید
- برای شناسایی نیاز به ورودی انسانی به `RequestInfoEvent` گوش کنید
- برای دریافت نتایج نهایی به `WorkflowOutputEvent` گوش کنید


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## مرحله 11: اجرای تست مورد 2 - شهر همراه با موجودی (استکهلم - نیازی به ورودی انسانی نیست)

این تست **مسیر مستقیم** را زمانی که اتاق‌ها در دسترس هستند، نشان می‌دهد:

1. درخواست هتل در استکهلم
2. بررسی availability_agent → اتاق‌ها در دسترس ✅
3. پیشنهاد رزرو توسط booking_agent
4. نمایش تاییدیه توسط display_result
5. **نیازی به ورودی انسانی نیست!**

جریان کاری مسیر انسان در حلقه را کاملاً زمانی که اتاق‌ها در دسترس هستند دور می‌زند.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## نکات کلیدی و بهترین شیوه‌ها در انسان-در-چرخه

### ✅ آنچه یاد گرفتید:

#### 1. **الگوی RequestInfoExecutor**
الگوی انسان-در-چرخه در چارچوب Microsoft Agent از سه جزء کلیدی استفاده می‌کند:
- `RequestInfoExecutor` - روند کار را متوقف می‌کند و رویدادها را صادر می‌کند
- `RequestInfoMessage` - کلاس پایه برای بار درخواست تایپ‌شده (زیرکلاس دهید!)
- `RequestResponse` - پاسخ‌های انسانی را با درخواست‌های اصلی مرتبط می‌کند

**درک مهم:**
- `RequestInfoExecutor` خودش ورودی جمع‌آوری نمی‌کند - فقط روند کار را متوقف می‌کند
- کد برنامه شما باید به `RequestInfoEvent` گوش دهد و ورودی جمع‌آوری کند
- باید با دیکشنری که `request_id` را به پاسخ کاربر نگاشت می‌کند، فراخوانی `send_responses_streaming()` انجام دهید

#### 2. **الگوی اجرای پخشی (Streaming)**
```python
# تکرار اول
stream = workflow.run_stream(initial_request)

# تکرارهای بعدی (بعد از ورودی انسان)
stream = workflow.send_responses_streaming(pending_responses)

# همیشه رویدادها را پردازش کن
events = [event async for event in stream]
```

#### 3. **معماری رویدادمحور**
به رویدادهای مشخص گوش دهید تا روند کار کنترل شود:
- `RequestInfoEvent` - ورودی انسانی نیاز است (روند کار متوقف شده)
- `WorkflowOutputEvent` - نتیجه نهایی آماده است (روند کار کامل شده)
- `WorkflowStatusEvent` - تغییرات وضعیت (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS و غیره)

#### 4. **اجراهای سفارشی با @handler**
`DecisionManager` نشان می‌دهد چگونه اجراگرهایی بسازیم که:
- از دکوراتور `@handler` برای نمایش متدها به‌عنوان مراحل روند کار استفاده می‌کنند
- پیام‌های تایپ‌شده را دریافت می‌کنند (مثلاً `RequestResponse[HumanFeedbackRequest, str]`)
- با ارسال پیام به اجراگرهای دیگر، روند کار را مسیریابی می‌کنند
- از طریق `WorkflowContext` به زمینه دسترسی دارند

#### 5. **مسیردهی شرطی با تصمیمات انسانی**
می‌توانید توابع شرطی بسازید که پاسخ‌های انسانی را ارزیابی کنند:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 کاربردهای دنیای واقعی:

1. **روندهای تصویب**
   - قبل از پردازش گزارش‌های هزینه، تأیید مدیر را دریافت کنید
   - قبل از ارسال ایمیل‌های خودکار، نیاز به بازبینی انسانی داشته باشید
   - قبل از اجرای تراکنش‌های ارزش بالا، تأیید کنید

2. **مدیریت محتوا**
   - محتوای مشکوک را برای بررسی انسانی علامت‌گذاری کنید
   - از مدیران بخواهید درباره موارد خاص تصمیم نهایی بگیرند
   - وقتی اعتماد AI پایین است، موضوع را به انسان ارجاع دهید

3. **خدمات به مشتری**
   - اجازه دهید AI به سوالات معمولی به طور خودکار پاسخ دهد
   - مسائل پیچیده را به نمایندگان انسانی ارجاع دهید
   - از مشتری بپرسید آیا می‌خواهد با انسان صحبت کند

4. **پردازش داده‌ها**
   - از انسان‌ها بخواهید ورودی‌های داده مبهم را حل کنند
   - تفسیرهای AI از اسناد نامشخص را تأیید کنید
   - به کاربران اجازه دهید بین تفاسیر معتبر مختلف انتخاب کنند

5. **سیستم‌های حساس به ایمنی**
   - قبل از اعمال غیرقابل بازگشت، تایید انسانی بخواهید
   - قبل از دسترسی به داده‌های حساس، تأییدیه بگیرید
   - تصمیمات را در صنایع مقرراتی (سلامت، مالی) تأیید کنید

6. **عوامل تعاملی**
   - ربات‌های مکالمه‌ای بسازید که سوالات پیگیری بپرسند
   - راهنماهایی ایجاد کنید که کاربران را در فرآیندهای پیچیده هدایت کنند
   - عامل‌هایی طراحی کنید که گام به گام با انسان‌ها همکاری کنند

### 🔄 مقایسه: با و بدون انسان-در-چرخه

| ویژگی | روند کار شرطی | روند کار انسان-در-چرخه |
|---------|---------------------|---------------------------|
| **اجرا** | یک‌بار `workflow.run()` | حلقه با `run_stream()` + `send_responses_streaming()` |
| **ورودی کاربر** | ندارد (کاملاً خودکار) | درخواست‌های تعاملی از طریق `input()` یا رابط کاربری |
| **اجزا** | عوامل + اجراگرها | + RequestInfoExecutor + DecisionManager |
| **رویدادها** | فقط AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent و غیره |
| **توقف روند کار** | توقف ندارد | روند کار در RequestInfoExecutor متوقف می‌شود |
| **کنترل انسانی** | کنترل انسانی ندارد | انسان‌ها تصمیمات کلیدی می‌گیرند |
| **کاربرد** | خودکارسازی | همکاری و نظارت |

### 🚀 الگوهای پیشرفته:

#### چندین نقطه تصمیم‌گیری انسانی
می‌توانید چندین گره `RequestInfoExecutor` در همان روند کار داشته باشید:
```python
.add_edge(agent1, request_info_1)  # نخستین تصمیم انسانی
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # دومین تصمیم انسانی
.add_edge(decision_manager_2, final_agent)
```

#### مدیریت تایم‌اوت
تایم‌اوت برای پاسخ‌های انسانی پیاده‌سازی کنید:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # گزینه پیش‌فرض ایمن است
```

#### یکپارچه‌سازی رابط کاربری غنی
به جای `input()`، با رابط‌های وب، Slack، Teams و غیره یکپارچه شوید:
```python
if isinstance(event, RequestInfoEvent):
    # ارسال اعلان به کانال ترجیحی کاربر
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### انسان-در-چرخه شرطی
فقط در شرایط خاص ورودی انسان بخواهید:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # فقط در صورت پایین بودن اطمینان یا بالا بودن مقدار به انسان ارسال شود
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ بهترین شیوه‌ها:

1. **همیشه از RequestInfoMessage زیرکلاس بسازید**
   - ایمنی نوع و اعتبارسنجی را فراهم می‌کند
   - زمینه غنی برای رندرینگ رابط کاربری را ممکن می‌سازد
   - هدف هر نوع درخواست را روشن می‌کند

2. **از درخواست‌های توصیفی استفاده کنید**
   - زمینه درخواست خود را بگنجانید
   - پیامدهای هر انتخاب را توضیح دهید
   - سوالات را ساده و واضح نگه دارید

3. **ورودی غیرمنتظره را مدیریت کنید**
   - پاسخ‌های کاربر را اعتبارسنجی کنید
   - مقدار پیش‌فرض برای ورودی نامعتبر فراهم کنید
   - پیام‌های خطای واضح بدهید

4. **شناسه‌های درخواست را پیگیری کنید**
   - از همبستگی بین request_id و پاسخ‌ها استفاده کنید
   - تلاش نکنید وضعیت را دستی مدیریت کنید

5. **برای غیرمسدودسازی طراحی کنید**
   - نخ‌ها را منتظر ورودی نگه ندارید
   - از الگوهای async در سراسر استفاده کنید
   - از نمونه‌های همزمان روند کار پشتیبانی کنید

### 📚 مفاهیم مرتبط:

- **میان‌افزار عامل** - رهگیری تماس‌های عامل (دفترچه قبلی)
- **مدیریت وضعیت روند کار** - حفظ وضعیت روند کار بین اجراها
- **همکاری چندعامله** - ترکیب انسان-در-چرخه با تیم‌های عامل
- **معماری‌های رویدادمحور** - ساخت سیستم‌های واکنشی با رویدادها

---

### 🎓 تبریک می‌گوییم!

شما روندهای انسان-در-چرخه را با چارچوب Microsoft Agent به خوبی فرا گرفتید! اکنون می‌دانید چگونه:
- ✅ روندهای کار را متوقف کنید تا ورودی انسانی جمع‌آوری شود
- ✅ از RequestInfoExecutor و RequestInfoMessage استفاده کنید
- ✅ اجرای پخشی را با رویدادها مدیریت کنید
- ✅ اجراگرهای سفارشی با @handler بسازید
- ✅ روندهای کار را بر اساس تصمیمات انسانی مسیریابی کنید
- ✅ عامل‌های AI تعاملی بسازید که با انسان‌ها همکاری کنند

**این یک الگوی حیاتی برای ساخت سیستم‌های AI قابل‌اعتماد و قابل کنترل است!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
